# Source-Free Adaptation for IoT IDS — Phase 2: is a benign anchor enough, and is the problem only the threshold?
Phase 1 showed that with *no* target information, adaptation fails in 12/12 directions and the target attack rate is not identifiable. The minimal extra information is a small **benign anchor** (flows the operator believes are normal). Before designing any method, this phase tests whether trivial uses of that anchor already solve the problem:

| Baseline | Uses source model | Uses benign anchor |
|---|---|---|
| source_only | yes | no |
| np_threshold — source attack score, threshold at the anchor's (1−α) quantile (Neyman–Pearson, α = 1%) | yes | yes |
| iforest — Isolation Forest trained on the anchor only | no | yes |
| autoencoder — trained on the anchor only, reconstruction error | no | yes |
| mahalanobis — distance to the anchor in the source model's feature space | yes | yes |
| oracle — supervised on labelled target | — | — |

**AUROC** (threshold-free) separates a *threshold* problem (high AUROC, wrong cut) from a *representation* problem (low AUROC).
Anchor size 100 / 1000 flows; contamination 0% / 5% attacks. Features: no_artifacts (stricter). 12 directions × 3 seeds × 2 priors.

**Decision rule (printed):** NO-GO for a method paper if a trivial baseline already recovers ≥ 70% of the gap in most directions. GO if every trivial baseline recovers < 50% in at least half of the directions.

In [1]:
# ===== 1. Setup =====
import os, glob, json, time, math, random, copy
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.ensemble import IsolationForest
try:
    from google.colab import drive
    drive.mount('/content/drive'); ROOT = '/content/drive/MyDrive'
except Exception:
    ROOT = '.'
OUT = f'{ROOT}/sfda_ids_phase2'; os.makedirs(OUT, exist_ok=True)
P1 = f'{ROOT}/sfda_ids_phase1'                       # reuse Phase 1 data samples
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, '| results ->', OUT)

CFG = dict(
    datasets={'ton': 'dhoogla/nftoniotv2', 'unsw': 'dhoogla/nfunswnb15v2', 'bot': 'dhoogla/nfbotiotv2', 'cic': 'dhoogla/nfcsecicids2018v2'},
    required=['ton', 'unsw'],
    attack_rate={'ton': 0.6399, 'unsw': 0.0398, 'bot': 0.9964, 'cic': 0.1195},
    n_per_dataset=300_000, test_frac=0.3, seeds=[0, 1, 2], hidden=[128, 64], src_epochs=5, bs=512, lr=1e-3,
    priors=['natural', 0.05], n_target_test=40_000, alpha=0.01,
    anchor_sizes=[100, 1000], contaminations=[0.0, 0.05],
    artifacts=['MIN_TTL', 'MAX_TTL', 'TCP_WIN_MAX_IN', 'TCP_WIN_MAX_OUT'], ae_steps=1500,
    trivial_ok=0.7, go_max=0.5)
json.dump(CFG, open(f'{OUT}/config.json', 'w'), indent=2)
DROP = ['IPV4_SRC_ADDR', 'IPV4_DST_ADDR', 'L4_SRC_PORT', 'L4_DST_PORT', 'DNS_QUERY_ID', 'Dataset']
NEG = ['0', '0.0', 'benign', 'false', 'normal']
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

Mounted at /content/drive
device: cuda | results -> /content/drive/MyDrive/sfda_ids_phase2


In [2]:
# ===== 2. Data (Phase 1 samples reused from Drive; otherwise downloaded) =====
def label_of(df):
    cols = {c.lower(): c for c in df.columns}; lab = cols.get('label')
    return (~df[lab].astype(str).str.strip().str.lower().isin(NEG)).values

def load_nf(key, slug, n, seed=0):
    for path in (f'{P1}/data_{key}_{n}.parquet', f'{OUT}/data_{key}_{n}.parquet'):
        if os.path.exists(path): print(key, 'loaded', path); return pd.read_parquet(path)
    import kagglehub
    root = kagglehub.dataset_download(slug)
    files = sorted(glob.glob(os.path.join(root, '**', '*.parquet'), recursive=True))
    import pyarrow.parquet as pq
    pfs = [pq.ParquetFile(f) for f in files]; tot = sum(p.metadata.num_rows for p in pfs)
    r = CFG['attack_rate'][key]; rs = np.random.RandomState(seed); parts = []
    for p in pfs:
        for b in p.iter_batches(batch_size=1_000_000):
            d = b.to_pandas(); y = label_of(d)
            pa = min(1.0, 0.65 * n / (tot * r)); pb = min(1.0, 0.65 * n / (tot * (1 - r)))
            parts.append(d[rs.rand(len(d)) < np.where(y, pa, pb)])
    df = pd.concat(parts, ignore_index=True)
    if len(df) > n: df = df.sample(n=n, random_state=seed).reset_index(drop=True)
    df.to_parquet(f'{OUT}/data_{key}_{n}.parquet'); return df

def prepare(df):
    cols = {c.lower(): c for c in df.columns}; lab, att = cols.get('label'), cols.get('attack')
    y = label_of(df).astype(np.int64)
    a = df[att].astype(str).str.strip().str.lower().values if att else np.where(y == 1, 'attack', 'benign')
    X = df.drop(columns=[c for c in [lab, att] + DROP if c is not None and c in df.columns])
    X = X.apply(lambda c: pd.to_numeric(c.astype(str) if str(c.dtype) in ('category', 'object', 'bool') else c, errors='coerce'))
    return X.loc[:, X.notna().any()].fillna(0.0), y, a

def build_data(raw):
    prep = {k: prepare(v) for k, v in raw.items()}
    common = sorted(set.intersection(*[set(p[0].columns) for p in prep.values()]))
    common = [c for c in common if c not in CFG['artifacts']]
    data = {}
    for k, (X, y, a) in prep.items():
        V = X[common].values; Xl = (np.sign(V) * np.log1p(np.abs(V))).astype(np.float32)
        idx = np.random.RandomState(0).permutation(len(y)); nt = int(CFG['test_frac'] * len(y))
        data[k] = dict(Xtr=Xl[idx[nt:]], ytr=y[idx[nt:]], Xte=Xl[idx[:nt]], yte=y[idx[:nt]])
    print(f'{len(common)} features (artifacts removed)'); return data, common

In [3]:
# ===== 3. Models and baselines =====
class Net(nn.Module):
    def __init__(s, d, h, k=2):
        super().__init__()
        s.f = nn.Sequential(nn.Linear(d, h[0]), nn.BatchNorm1d(h[0]), nn.ReLU(), nn.Linear(h[0], h[1]), nn.BatchNorm1d(h[1]), nn.ReLU())
        s.h = nn.Linear(h[1], k)
    def forward(s, x): return s.h(s.f(x))
def T(x): return torch.as_tensor(x, device=device)

def train_sup(X, y, seed, epochs):
    set_seed(seed); m = Net(X.shape[1], CFG['hidden']).to(device); opt = torch.optim.Adam(m.parameters(), CFG['lr'])
    Xt, yt = T(X), T(y); g = torch.Generator().manual_seed(seed)
    for _ in range(epochs):
        m.train(); perm = torch.randperm(len(Xt), generator=g).to(device)
        for i in range(0, len(perm), CFG['bs']):
            b = perm[i:i + CFG['bs']]
            if len(b) > 1: opt.zero_grad(); F.cross_entropy(m(Xt[b]), yt[b]).backward(); opt.step()
    return m.eval()

@torch.no_grad()
def attack_score(m, X):
    m.eval(); Xt = T(X)
    return torch.cat([F.softmax(m(Xt[i:i + 8192]), 1)[:, 1] for i in range(0, len(Xt), 8192)]).cpu().numpy()

@torch.no_grad()
def feats(m, X):
    m.eval(); Xt = T(X)
    return torch.cat([m.f(Xt[i:i + 8192]) for i in range(0, len(Xt), 8192)]).cpu().numpy().astype(np.float64)

def mahalanobis_score(Fa, Fe):
    mu = Fa.mean(0); C = np.cov(Fa, rowvar=False) + 1e-3 * np.eye(Fa.shape[1])
    Ci = np.linalg.pinv(C); D = Fe - mu
    return np.einsum('ij,jk,ik->i', D, Ci, D)

def ae_score(Xa, Xe, seed):
    set_seed(seed); d = Xa.shape[1]
    ae = nn.Sequential(nn.Linear(d, 32), nn.ReLU(), nn.Linear(32, 8), nn.ReLU(), nn.Linear(8, 32), nn.ReLU(), nn.Linear(32, d)).to(device)
    opt = torch.optim.Adam(ae.parameters(), 1e-3, weight_decay=1e-5); A = T(Xa)
    for _ in range(max(1, CFG['ae_steps'] // math.ceil(len(A) / 64))):
        perm = torch.randperm(len(A), device=device)
        for i in range(0, len(A), 64):
            b = perm[i:i + 64]; opt.zero_grad(); F.mse_loss(ae(A[b]), A[b]).backward(); opt.step()
    ae.eval()
    with torch.no_grad():
        E = T(Xe); err = torch.cat([((ae(E[i:i + 8192]) - E[i:i + 8192]) ** 2).mean(1) for i in range(0, len(E), 8192)])
        ea = ((ae(A) - A) ** 2).mean(1)
    return ea.cpu().numpy(), err.cpu().numpy()

def metrics(y, pred, score=None):
    pos, neg = y == 1, y == 0
    tpr = ((pred == 1) & pos).sum() / max(pos.sum(), 1); fpr = ((pred == 1) & neg).sum() / max(neg.sum(), 1)
    auc = roc_auc_score(y, score) if score is not None and 0 < y.mean() < 1 else np.nan
    return dict(macro_f1=f1_score(y, pred, average='macro', zero_division=0), attack_recall=float(tpr), fpr=float(fpr), auroc=float(auc))

def resample(X, y, prior, n, seed):
    rs = np.random.RandomState(seed); att, ben = np.where(y == 1)[0], np.where(y == 0)[0]
    n = int(min(n, len(att) / prior, len(ben) / (1 - prior))); na = int(round(prior * n))
    idx = rs.permutation(np.concatenate([rs.choice(att, na, replace=False), rs.choice(ben, n - na, replace=False)]))
    return X[idx], y[idx]

def make_anchor(X, y, size, contam, seed):
    rs = np.random.RandomState(seed); na = int(round(contam * size))
    ben, att = np.where(y == 0)[0], np.where(y == 1)[0]
    idx = np.concatenate([rs.choice(ben, size - na, replace=False), rs.choice(att, na, replace=False) if na else np.array([], int)])
    return X[idx]

In [4]:
# ===== 4. Runner (resumable) =====
def run_all(data, pairs, out_prefix, seeds):
    csv = f'{out_prefix}_metrics.csv'; done = set()
    if os.path.exists(csv):
        d = pd.read_csv(csv); done = set(zip(d.seed, d.src, d.tgt, d.prior.astype(str)))
    a = CFG['alpha']
    for seed in seeds:
        for s, t in pairs:
            todo = [p for p in CFG['priors'] if (seed, s, t, str(p)) not in done]
            if not todo: continue
            S, Tg = data[s], data[t]
            mu, sd = S['Xtr'].mean(0), S['Xtr'].std(0) + 1e-6
            norm = lambda X: np.clip((X - mu) / sd, -10, 10).astype(np.float32)
            t0 = time.time(); src = train_sup(norm(S['Xtr']), S['ytr'], seed, CFG['src_epochs'])
            for prior in todo:
                pe = CFG['attack_rate'][t] if prior == 'natural' else prior
                Xe, ye = resample(Tg['Xte'], Tg['yte'], pe, CFG['n_target_test'], seed + 100); Xe = norm(Xe)
                s_src = attack_score(src, Xe); F_e = feats(src, Xe)
                rows = [dict(method='source_only', anchor=0, contam=0.0, **metrics(ye, (s_src > 0.5).astype(int), s_src))]
                Xo, yo = resample(Tg['Xtr'], Tg['ytr'], pe, 100_000, seed)
                orc = train_sup(norm(Xo), yo, seed, CFG['src_epochs']); s_o = attack_score(orc, Xe)
                rows.append(dict(method='oracle', anchor=0, contam=0.0, **metrics(ye, (s_o > 0.5).astype(int), s_o)))
                for size in CFG['anchor_sizes']:
                    for c in CFG['contaminations']:
                        A = norm(make_anchor(Tg['Xtr'], Tg['ytr'], size, c, seed))
                        sa = attack_score(src, A); th = np.quantile(sa, 1 - a)
                        rows.append(dict(method='np_threshold', anchor=size, contam=c, **metrics(ye, (s_src > th).astype(int), s_src)))
                        iso = IsolationForest(n_estimators=200, random_state=seed).fit(A)
                        ia, ie = -iso.score_samples(A), -iso.score_samples(Xe)
                        rows.append(dict(method='iforest', anchor=size, contam=c, **metrics(ye, (ie > np.quantile(ia, 1 - a)).astype(int), ie)))
                        ea, ee = ae_score(A, Xe, seed)
                        rows.append(dict(method='autoencoder', anchor=size, contam=c, **metrics(ye, (ee > np.quantile(ea, 1 - a)).astype(int), ee)))
                        F_a = feats(src, A); ma, me = mahalanobis_score(F_a, F_a), mahalanobis_score(F_a, F_e)
                        rows.append(dict(method='mahalanobis', anchor=size, contam=c, **metrics(ye, (me > np.quantile(ma, 1 - a)).astype(int), me)))
                for r in rows: r.update(seed=seed, src=s, tgt=t, prior=str(prior), true_attack_rate=float(ye.mean()))
                pd.DataFrame(rows).to_csv(csv, mode='a', header=not os.path.exists(csv), index=False)
                brief = {f"{r['method']}@{r['anchor']}/{r['contam']}": round(r['macro_f1'], 3) for r in rows if r['contam'] == 0.0}
                print(f'seed {seed} {s}->{t} prior {prior}: {brief} ({(time.time()-t0)/60:.1f} min)', flush=True)
    return pd.read_csv(csv)

## Self-test on synthetic data — must print SELF-TEST PASSED

In [5]:
# ===== 5. Self-test =====
def synth(n, shift, prior, seed):
    rs = np.random.RandomState(seed); y = (rs.rand(n) < prior).astype(np.int64)
    return rs.randn(n, 8).astype(np.float32) + y[:, None] * 1.5 + shift, y
fake = {}
for i, (k, sh, pr) in enumerate([('A', 0.0, 0.6), ('B', 0.8, 0.3)]):
    X, y = synth(8000, sh, pr, i); fake[k] = dict(Xtr=X[2000:], ytr=y[2000:], Xte=X[:2000], yte=y[:2000])
_save = copy.deepcopy(CFG); CFG.update(n_target_test=1000, attack_rate={'A': 0.6, 'B': 0.3}, ae_steps=20, src_epochs=2)
for f in glob.glob(f'{OUT}/selftest_*'): os.remove(f)
mt = run_all(fake, [('A', 'B')], f'{OUT}/selftest', [0])
CFG.clear(); CFG.update(_save)
assert len(mt) == 2 * (2 + 4 * 2 * 2), f'self-test incomplete: {len(mt)} rows'
print('SELF-TEST PASSED'); print(mt[['prior', 'method', 'anchor', 'contam', 'macro_f1', 'fpr', 'auroc']].round(3).to_string(index=False))

seed 0 A->B prior natural: {'source_only@0/0.0': 0.663, 'oracle@0/0.0': 0.954, 'np_threshold@100/0.0': 0.968, 'iforest@100/0.0': 0.89, 'autoencoder@100/0.0': 0.881, 'mahalanobis@100/0.0': 0.431, 'np_threshold@1000/0.0': 0.968, 'iforest@1000/0.0': 0.894, 'autoencoder@1000/0.0': 0.861, 'mahalanobis@1000/0.0': 0.931} (0.1 min)
seed 0 A->B prior 0.05: {'source_only@0/0.0': 0.438, 'oracle@0/0.0': 0.727, 'np_threshold@100/0.0': 0.969, 'iforest@100/0.0': 0.85, 'autoencoder@100/0.0': 0.797, 'mahalanobis@100/0.0': 0.24, 'np_threshold@1000/0.0': 0.969, 'iforest@1000/0.0': 0.859, 'autoencoder@1000/0.0': 0.825, 'mahalanobis@1000/0.0': 0.868} (0.2 min)
SELF-TEST PASSED
  prior       method  anchor  contam  macro_f1   fpr  auroc
natural  source_only       0    0.00     0.663 0.480  0.999
natural       oracle       0    0.00     0.954 0.056  0.999
natural np_threshold     100    0.00     0.968 0.003  0.999
natural      iforest     100    0.00     0.890 0.013  0.969
natural  autoencoder     100    0.0

## Real data

In [6]:
# ===== 6. Load and run =====
raw = {}
for k, slug in CFG['datasets'].items():
    try: raw[k] = load_nf(k, slug, CFG['n_per_dataset'])
    except Exception as e:
        if k in CFG['required']: raise
        print('SKIPPED', k, e)
DATA, FEATS = build_data(raw); del raw
keys = list(DATA); PAIRS = [(s, t) for s in keys for t in keys if s != t]
MT = run_all(DATA, PAIRS, f'{OUT}/real', CFG['seeds'])
print('done:', len(MT), 'rows')

ton loaded /content/drive/MyDrive/sfda_ids_phase1/data_ton_300000.parquet
unsw loaded /content/drive/MyDrive/sfda_ids_phase1/data_unsw_300000.parquet
bot loaded /content/drive/MyDrive/sfda_ids_phase1/data_bot_300000.parquet
cic loaded /content/drive/MyDrive/sfda_ids_phase1/data_cic_300000.parquet
34 features (artifacts removed)
seed 0 ton->unsw prior natural: {'source_only@0/0.0': 0.361, 'oracle@0/0.0': 0.965, 'np_threshold@100/0.0': 0.487, 'iforest@100/0.0': 0.508, 'autoencoder@100/0.0': 0.669, 'mahalanobis@100/0.0': 0.511, 'np_threshold@1000/0.0': 0.488, 'iforest@1000/0.0': 0.59, 'autoencoder@1000/0.0': 0.804, 'mahalanobis@1000/0.0': 0.851} (0.4 min)
seed 0 ton->unsw prior 0.05: {'source_only@0/0.0': 0.37, 'oracle@0/0.0': 0.971, 'np_threshold@100/0.0': 0.485, 'iforest@100/0.0': 0.506, 'autoencoder@100/0.0': 0.694, 'mahalanobis@100/0.0': 0.532, 'np_threshold@1000/0.0': 0.485, 'iforest@1000/0.0': 0.588, 'autoencoder@1000/0.0': 0.818, 'mahalanobis@1000/0.0': 0.87} (0.8 min)
seed 0 ton->

In [7]:
# ===== 7. Analysis and decision =====
MT = pd.read_csv(f'{OUT}/real_metrics.csv'); MT['prior'] = MT.prior.astype(str)
MT['variant'] = np.where(MT.anchor > 0, MT.method + '@' + MT.anchor.astype(str) + '/c' + MT.contam.astype(str), MT.method)
G = MT.groupby(['src', 'tgt', 'prior', 'variant'])[['macro_f1', 'fpr', 'attack_recall', 'auroc']].mean()
F1 = G.macro_f1.unstack('variant'); gap = F1['oracle'] - F1['source_only']
REC = F1.sub(F1['source_only'], axis=0).div(gap.where(gap > 0.05), axis=0)
show = [c for c in REC.columns if c not in ('oracle', 'source_only')]
print('Fraction of the gap recovered:'); print(REC[show].round(2).to_string())

print('\nMedian over directions — recovery, FPR (target alpha = 1%), attack recall, AUROC:')
summ = pd.DataFrame({'median_recovery': REC[show].median(),
                     'n_dirs_recovery>=0.7': (REC[show] >= CFG['trivial_ok']).sum(),
                     'n_dirs_recovery<0.5': (REC[show] < CFG['go_max']).sum(),
                     'median_fpr': G.fpr.unstack('variant')[show].median(),
                     'median_attack_recall': G.attack_recall.unstack('variant')[show].median(),
                     'median_auroc': G.auroc.unstack('variant')[show].median()}).round(3)
summ.loc['source_only', 'median_auroc'] = round(G.auroc.unstack('variant')['source_only'].median(), 3)
summ.loc['oracle', 'median_auroc'] = round(G.auroc.unstack('variant')['oracle'].median(), 3)
print(summ.to_string()); summ.to_csv(f'{OUT}/summary.csv')

AUC = G.auroc.unstack('variant')[['source_only', 'oracle']].round(3)
print('\nSource-model AUROC on target (threshold-free ranking quality):'); print(AUC.to_string())

clean = [c for c in show if c.endswith('c0.0')]
best_clean = REC[clean].max(1)
n = len(best_clean.dropna())
trivial = (best_clean >= CFG['trivial_ok']).sum() > n / 2
go = (best_clean < CFG['go_max']).sum() >= n / 2
dec = dict(directions_x_priors=int(n), best_trivial_median_recovery=round(float(best_clean.median()), 3),
           n_best_trivial_recovery_ge_0_7=int((best_clean >= CFG['trivial_ok']).sum()),
           n_best_trivial_recovery_lt_0_5=int((best_clean < CFG['go_max']).sum()),
           source_auroc_median=round(float(AUC['source_only'].median()), 3),
           decision='NO-GO: a trivial use of the benign anchor already solves most cases' if trivial
                    else 'GO: trivial baselines fail -> a representation-level method is needed' if go
                    else 'MIXED: trivial baselines solve some directions only -> inspect per direction')
json.dump(dec, open(f'{OUT}/decision.json', 'w'), indent=2); print('\n', json.dumps(dec, indent=2))

Fraction of the gap recovered:
variant            autoencoder@100/c0.0  autoencoder@100/c0.05  autoencoder@1000/c0.0  autoencoder@1000/c0.05  iforest@100/c0.0  iforest@100/c0.05  iforest@1000/c0.0  iforest@1000/c0.05  mahalanobis@100/c0.0  mahalanobis@100/c0.05  mahalanobis@1000/c0.0  mahalanobis@1000/c0.05  np_threshold@100/c0.0  np_threshold@100/c0.05  np_threshold@1000/c0.0  np_threshold@1000/c0.05
src  tgt  prior                                                                                                                                                                                                                                                                                                                                                                      
bot  cic  0.05                     0.36                   0.15                   0.44                    0.14              0.25               0.23               0.30                0.19                  0.12              